# Chapter 9: Recompute paired uncertainty and inspect coverage

**The question.** A discovery becomes a finding only when it holds on fresh data with honest uncertainty. The Chapter 9 companion in `regional_case` does this for the historical run. This notebook repeats the key calculation for the live run: paired comparisons between search methods on its new confirmation sample.

**What was run live.** In this run the language model proposed a different change from the historical run: a new feature, the product of business-receipt share and cash intensity, instead of merging clusters. The search that uses that feature is the `reflection` arm.

**Result.** The new feature beat continued search at one of three seeds (74101) and could not be told apart from it at the other two. Twinning beat random starts at one seed (74103); at the other two both froze the same group.

**Terms.** *Contrast*: how much worse the model does inside a group than outside it; higher means a weaker spot was found.

**Run.** Run 20260923T220909Z repeated the whole regional study with new optimizer seeds (74101 to 74103), a new language-model proposal and a new 200,000-customer confirmation sample, then ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. Its numbers should not be mixed with those of the `regional_case` notebooks. This notebook reads the recorded calls; rerunning it makes no new call.

## Setup and source checks

Loads this run's records and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Recompute the paired comparisons

Each comparison measures two frozen groups on the same 200,000 new customers and takes the difference in their contrasts. Using the same customers for both removes the noise the two groups share; each customer's contribution to that difference is

$$\psi_i=\frac{m_i(L_i-\bar L_m)}{\pi}-\frac{(1-m_i)(L_i-\bar L_{\neg m})}{1-\pi}.$$

The ranges are widened with a Bonferroni correction, $z=\Phi^{-1}(1-0.05/38)$, for the study's 19 claims. The cell checks each recomputed range against the saved one, then shows the live answer verdicts.

In [2]:
from scipy.stats import norm
a=np.load(R/'regional/confirmation.npz');loss=(a['y']-a['p'])**2
z=norm.ppf(1-.05/(2*19))
def influence(key):
 m=a[key];p=m.mean()
 return m*(loss-loss[m].mean())/p-(~m)*(loss-loss[~m].mean())/(1-p)
for row in results['comparisons']:
 ka=f"{row['a']}_{row['seed']}";kb=f"{row['b']}_{row['seed']}"
 diff=results['confirmation'][ka]['contrast']-results['confirmation'][kb]['contrast']
 se=(influence(ka)-influence(kb)).std(ddof=1)/np.sqrt(len(loss))
 assert np.allclose([diff-z*se,diff+z*se],row['simultaneous_ci'])
display(pd.DataFrame(results['comparisons']))
rows=pd.DataFrame(read('agent_results.json')['rows'])
display(rows.groupby(['phase','verdict']).size())
print(results['scope'])

,seed,a,b,contrast_difference,standard_error,simultaneous_ci
0,74101,reflection,continuation,0.012834,0.001590,"[0.008051940751484483, 0.017616414017858083]"
1,74101,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
2,74102,reflection,continuation,-0.001141,0.001139,"[-0.0045669455524634285, 0.0022844287437013173]"
3,74102,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
4,74103,reflection,continuation,-0.000902,0.001287,"[-0.004772999696062933, 0.002968597897497625]"
5,74103,twinning,random,0.006413,0.001018,"[0.003349358407588642, 0.009475817055983117]"


phase      verdict           
discovery  correct_abstention    36
           correct_answer        28
reserved   correct_abstention     3
           correct_answer         3
dtype: int64

Three optimizer seeds share one discovery sample, one reflection and one confirmation sample; not independent study replications.


**Read the output.** **In one line: the new feature helped at one seed out of three, and all 70 live answers were correct.**

**Columns.** First table: `seed` is the run; `a` and `b` are the two methods compared (`reflection` is the search with the language model's new feature); `contrast_difference` is a minus b, where positive means a found a weaker spot; `standard_error` is its uncertainty; `simultaneous_ci` is the range of plausible values, and a range that includes zero means the methods cannot be told apart. Second table: the live answer verdicts from Chapter 8, by phase.

**Notice.**

- New feature against continuation: +0.012834 at seed 74101, range [0.008052, 0.017616], above zero. At 74102 and 74103 the ranges include zero.
- Twinning against random: +0.006413 at seed 74103, range [0.003349, 0.009476]. At the other two seeds the difference is exactly zero because both methods froze the same group.
- Answers: 36 correct declines and 28 correct answers on discovery, 3 of each on reserved, with no wrong, unsupported or unscorable output.
- The three seeds share one discovery sample, one proposal and one confirmation sample, so they are not independent replications.